# Notebook 02 – Implementação de Métricas Avançadas de Drift

**Aula 04 – Métricas Avançadas para Detecção de Drift** (Vídeos 2 e 3)

## Objetivos

1. Revisar o PSI e demonstrar sua limitação para drift multivariado.
2. Implementar passo a passo a **Maximum Mean Discrepancy (MMD)** com kernel RBF.
3. Calcular **Wasserstein Distance** e **Energy Distance**.
4. Comparar a sensibilidade de todas as métricas em dados sintéticos 2-D (inversão de correlação renda–idade).
5. Calibrar limiares via validação cruzada e ajustar o hiperparâmetro γ do kernel RBF.

### Base Teórica (Documento 04)

Conforme o **Documento 04**, a MMD é definida como:

$$\text{MMD}^2(P,Q) = \mathbb{E}[k(x,x')] + \mathbb{E}[k(y,y')] - 2\,\mathbb{E}[k(x,y)]$$

onde $k$ é um kernel RBF: $k(x,x') = \exp(-\gamma \|x - x'\|^2)$.

A MMD é zero se e somente se $P = Q$ (Gretton et al., 2012). Para dados de alta dimensão,
a MMD captura diferenças nas distribuições conjuntas que métricas univariadas como o PSI não conseguem detectar.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

# Adiciona a raiz do projeto ao path para importar os módulos src
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_preprocessing import DataPreprocessor
from src.model import (
    PSICalculator,
    MMDCalculator,
    WassersteinCalculator,
    EnergyDistanceCalculator,
    DriftDetector,
)
from src.training import train_model, cross_validate, hyperparameter_tuning
from src.evaluation import (
    calculate_metrics,
    plot_distributions,
    plot_scatter_drift,
    plot_metric_comparison,
    plot_correlation_heatmaps,
)
from src.utils import save_model, save_metrics

print(f"Raiz do projeto: {PROJECT_ROOT}")

## 1. Carregamento dos Dados

Carregamos o dataset sintético gerado no **Notebook 01** (exploração inicial).
O dataset contém dois períodos — *referência* e *atual* — com inversão de correlação
renda–idade no período atual, conforme descrito no **Documento 04**.

In [ ]:
# Carrega e prepara o dataset
preprocessor = DataPreprocessor(n_samples=5000, seed=42)
df = preprocessor.load_data()

# Separa em referência e atual
df_ref, df_cur = preprocessor.split_data(df)

# Normaliza as features
df_ref_prep = preprocessor.prepare_features(df_ref, normalize=True)
df_cur_prep = preprocessor.prepare_features(df_cur, normalize=True)

# Converte para arrays numpy
X_ref = df_ref_prep.values
X_cur = df_cur_prep.values
feature_names = list(df_ref_prep.columns)

print(f"Shape referência: {X_ref.shape}")
print(f"Shape atual:      {X_cur.shape}")
print(f"Features:         {feature_names}")

## 2. PSI – Population Stability Index (Revisão)

Conforme o **Documento 04** (Vídeo 2, Snippet 1 do Hands On), o PSI é definido como:

$$\text{PSI} = \sum_{i=1}^{B} (p_i - q_i) \cdot \ln\!\left(\frac{p_i}{q_i}\right)$$

O PSI é uma métrica **univariada** — calcula-se por feature individualmente.
Threshold convencional: **PSI > 0.25** indica drift significativo.

> **Limitação**: o PSI não detecta mudanças na *estrutura de correlação* entre variáveis,
> apenas shifts marginais. Isso será demonstrado na comparação com a MMD.

In [ ]:
# Calcula PSI por feature
psi_calc = PSICalculator(n_bins=10)
psi_values = psi_calc.calculate_multifeature(X_ref, X_cur)

# Visualização: gráfico de barras com linha de threshold
fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(feature_names, psi_values, color="steelblue", edgecolor="white")
ax.axhline(y=0.25, color="red", linestyle="--", linewidth=1.5, label="Threshold (0.25)")
ax.set_xlabel("Feature")
ax.set_ylabel("PSI")
ax.set_title("PSI por Feature (Documento 04 – Vídeo 3)")
ax.legend()

# Anotação nos valores
for bar, val in zip(bars, psi_values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
            f"{val:.4f}", ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.show()

# Interpretação
for name, psi in zip(feature_names, psi_values):
    status = "DRIFT" if psi > 0.25 else "OK"
    print(f"  {name:>15s}: PSI = {psi:.4f}  [{status}]")

print("\n→ O PSI pode não acusar drift mesmo quando a correlação entre features se inverte.")

## 3. Maximum Mean Discrepancy (MMD)

Conforme o **Documento 04** (Vídeo 2) e Gretton et al. (2012), a MMD mede a distância
entre duas distribuições no espaço RKHS (Reproducing Kernel Hilbert Space):

$$\text{MMD}^2(P,Q) = \mathbb{E}[k(x,x')] + \mathbb{E}[k(y,y')] - 2\,\mathbb{E}[k(x,y)]$$

Com kernel **RBF** (Gaussiano):

$$k(x,x') = \exp(-\gamma \|x - x'\|^2)$$

**Vantagens sobre o PSI**:
- Métrica **multivariada**: captura mudanças na distribuição conjunta.
- Detecta inversão de correlação e outros drifts estruturais.
- Formulação matemática rigorosa com garantias teóricas.

A seguir, implementamos a MMD passo a passo (como no Snippet 2 do Hands On do
**Documento 04**) e comparamos com o `MMDCalculator` do módulo `src.model`.

In [ ]:
# ------------------------------------------------------------------
# Implementação manual da MMD — passo a passo (Snippet 2, Documento 04)
# ------------------------------------------------------------------
from scipy.spatial.distance import cdist

# 1. Seleção de γ via mediana heuristic (Gretton et al., 2012)
combined = np.vstack([X_ref, X_cur])
dists = cdist(combined[:1000], combined[:1000], metric="sqeuclidean")
median_dist = np.median(dists[dists > 0])
gamma = 1.0 / (2.0 * median_dist)
print(f"γ (mediana heuristic): {gamma:.6f}")

# 2. Matrizes de kernel
K_XX = np.exp(-gamma * cdist(X_ref, X_ref, metric="sqeuclidean"))
K_YY = np.exp(-gamma * cdist(X_cur, X_cur, metric="sqeuclidean"))
K_XY = np.exp(-gamma * cdist(X_ref, X_cur, metric="sqeuclidean"))

print(f"\nDimensões das matrizes de kernel:")
print(f"  K_XX: {K_XX.shape}  (mean = {K_XX.mean():.6f})")
print(f"  K_YY: {K_YY.shape}  (mean = {K_YY.mean():.6f})")
print(f"  K_XY: {K_XY.shape}  (mean = {K_XY.mean():.6f})")

# 3. Cálculo da MMD²
mmd_sq_manual = K_XX.mean() + K_YY.mean() - 2 * K_XY.mean()
print(f"\nMMD² (manual):  {mmd_sq_manual:.6f}")

# 4. Comparação com MMDCalculator do src.model
mmd_calc = MMDCalculator(gamma=None)  # usa mediana heuristic internamente
mmd_sq_src = mmd_calc.calculate(X_ref, X_cur)
print(f"MMD² (src.model): {mmd_sq_src:.6f}")
print(f"\n→ Os valores devem ser próximos (diferenças por amostragem na heurística).")

### 3.1 Teste de Permutação para MMD

Conforme o **Documento 04** (Vídeo 2, Seção "Saiba Mais"), para determinar se o valor
observado de MMD é estatisticamente significativo, aplicamos um **teste de permutação**
(Gretton et al., 2012):

1. Combinar as duas amostras.
2. Permutar aleatoriamente e dividir em dois grupos do mesmo tamanho.
3. Calcular MMD² para cada permutação → distribuição nula.
4. Comparar a MMD observada com o percentil 95 da distribuição nula.

Se $p\text{-valor} < 0.05$, rejeitamos $H_0: P = Q$ e concluímos que há drift.

In [ ]:
# Teste de permutação usando MMDCalculator
mmd_calc = MMDCalculator(gamma=None)
result = mmd_calc.permutation_test(X_ref, X_cur, n_permutations=100, seed=42)

print("=== Teste de Permutação para MMD (Documento 04) ===")
print(f"  MMD² observada:   {result.statistic:.6f}")
print(f"  Threshold (p95):  {result.threshold:.6f}")
print(f"  p-valor:          {result.details['p_value']:.4f}")
print(f"  Drift detectado:  {result.drift_detected}")

# Histograma da distribuição nula
# Recalculamos as MMDs nulas para visualização
rng = np.random.RandomState(42)
combined = np.vstack([X_ref, X_cur])
n = len(X_ref)
null_mmds = []
for _ in range(100):
    perm = rng.permutation(len(combined))
    null_mmds.append(mmd_calc.calculate(combined[perm[:n]], combined[perm[n:]]))

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(null_mmds, bins=25, alpha=0.7, color="skyblue", edgecolor="white",
        label="Distribuição Nula (H₀)")
ax.axvline(result.statistic, color="red", linewidth=2,
           label=f"MMD² observada = {result.statistic:.4f}")
ax.axvline(result.threshold, color="orange", linewidth=1.5, linestyle="--",
           label=f"Threshold p95 = {result.threshold:.4f}")
ax.set_xlabel("MMD²")
ax.set_ylabel("Frequência")
ax.set_title("Teste de Permutação – Distribuição Nula vs MMD Observada")
ax.legend()
plt.tight_layout()
plt.show()

## 4. Distância de Wasserstein (Earth Mover's Distance)

Conforme o **Documento 04** (Vídeo 2), a distância de Wasserstein mede o "custo mínimo
de transporte" para transformar uma distribuição em outra. Para distribuições
univariadas, equivale à **área entre as CDFs**:

$$W_1(P,Q) = \int_{-\infty}^{\infty} |F_P(x) - F_Q(x)|\, dx$$

Referências: Arjovsky et al. (2017) — Wasserstein GAN; Villani (2009) — Optimal Transport.

Assim como o PSI, a Wasserstein calculada feature a feature é **univariada**, mas fornece
uma interpretação geométrica mais intuitiva ("custo de transporte").

In [ ]:
# Calcula Wasserstein por feature
wass_calc = WassersteinCalculator()
wass_values = wass_calc.calculate_multifeature(X_ref, X_cur)

print("Distância de Wasserstein por feature (Documento 04 – Vídeo 2):")
for name, w in zip(feature_names, wass_values):
    print(f"  {name:>15s}: W₁ = {w:.4f}")

print(f"\n  Média:           W₁ = {np.mean(wass_values):.4f}")

## 5. Energy Distance

Conforme o **Documento 04** (Vídeo 2), a Energy Distance é uma métrica **multivariada**
baseada em distâncias euclidianas entre observações (Székely & Rizzo, 2013):

$$\mathcal{E}(X,Y) = 2\,\mathbb{E}\|X - Y\| - \mathbb{E}\|X - X'\| - \mathbb{E}\|Y - Y'\|$$

A Energy Distance é zero se e somente se $P = Q$, assim como a MMD.
É equivalente ao **teste de Cramér** em contextos univariados.

In [ ]:
# Calcula Energy Distance
energy_calc = EnergyDistanceCalculator()
energy_dist = energy_calc.calculate(X_ref, X_cur)

print(f"Energy Distance (Documento 04 – Székely & Rizzo, 2013): {energy_dist:.6f}")
print(f"\n→ Valor > 0 indica divergência entre as distribuições conjuntas.")

## 6. Comparação de Sensibilidade: PSI vs MMD

Conforme o **Documento 04** (Vídeo 3), quando há inversão de correlação entre features
(ex.: renda–idade), as distribuições marginais podem permanecer inalteradas.
Nesse cenário:

- **PSI** (univariado): pode **não** detectar drift.
- **MMD** e **Energy Distance** (multivariados): detectam a mudança na distribuição conjunta.

Consolidamos todas as métricas calculadas e visualizamos a comparação.

In [ ]:
# Calcula todas as métricas de uma vez
metrics = calculate_metrics(X_ref, X_cur, feature_names=feature_names)

# Visualização comparativa
fig = plot_metric_comparison(metrics)
plt.show()

# Tabela consolidada
print("\n" + "=" * 60)
print("COMPARAÇÃO DE MÉTRICAS (Documento 04 – Vídeo 3)")
print("=" * 60)
print(f"{'Métrica':<25s} {'Valor':>10s}  {'Detecta Drift?':>15s}")
print("-" * 60)
print(f"{'PSI (máx por feature)':<25s} {metrics['psi']['max']:>10.4f}  "
      f"{'SIM' if metrics['psi']['max'] > 0.25 else 'NÃO':>15s}")
print(f"{'PSI (média)':<25s} {metrics['psi']['mean']:>10.4f}  "
      f"{'SIM' if metrics['psi']['mean'] > 0.25 else 'NÃO':>15s}")
print(f"{'MMD²':<25s} {metrics['mmd']:>10.4f}  "
      f"{'SIM (valor > 0)':>15s}")
print(f"{'Wasserstein (média)':<25s} {metrics['wasserstein']['mean']:>10.4f}")
print(f"{'Energy Distance':<25s} {metrics['energy_distance']:>10.4f}  "
      f"{'SIM (valor > 0)':>15s}")
print("=" * 60)
print("\n→ PSI pode falhar para drift multivariado; MMD e Energy Distance são mais sensíveis.")

## 7. Calibração de Limiares via Validação Cruzada

Conforme o **Documento 04**, para definir limiares de significância estatística de forma
robusta, utilizamos **validação cruzada** nos dados de referência:

- Dividimos os dados de referência em *folds*.
- Para cada fold, calculamos a MMD entre os subconjuntos (sob $H_0$: mesma distribuição).
- O **percentil 95** da distribuição resultante define o limiar de detecção.

Isso evita calibração ad-hoc e fornece um threshold adaptado aos dados.

In [ ]:
# Validação cruzada nos dados de referência
cv_results = cross_validate(X_ref, n_folds=5, n_permutations=100, seed=42)

print("=== Calibração de Limiares via Validação Cruzada (Documento 04) ===")
print(f"  Threshold (p95): {cv_results['threshold_95']:.6f}")
print(f"  Média nula:      {cv_results['mean_null']:.6f}")
print(f"  Desvio padrão:   {cv_results['std_null']:.6f}")
print(f"  Nº de MMDs nulas: {len(cv_results['null_mmds'])}")

# Comparação: MMD observada vs threshold calibrado
mmd_observed = mmd_calc.calculate(X_ref, X_cur)
print(f"\n  MMD² observada:  {mmd_observed:.6f}")
print(f"  Drift detectado: {mmd_observed > cv_results['threshold_95']}")

## 8. Busca de Hiperparâmetros (γ do Kernel RBF)

Conforme o **Documento 04** e Gretton et al. (2012), a escolha de γ afeta diretamente
a sensibilidade da MMD:

- **γ muito pequeno**: kernel muito "largo", perde sensibilidade local.
- **γ muito grande**: kernel muito "estreito", sensível a ruído.
- **Mediana heuristic**: compromisso robusto — $\gamma = 1 / (2 \cdot \text{mediana}^2)$.

Buscamos o γ que maximiza a **separação** entre a MMD observada e a distribuição nula.

In [ ]:
# Busca de hiperparâmetros para γ
tuning_results = hyperparameter_tuning(X_ref, X_cur, seed=42)

print("=== Busca de Hiperparâmetros – γ do Kernel RBF (Documento 04) ===")
print(f"  Melhor γ:         {tuning_results['best_gamma']:.6f}")
print(f"  Melhor separação: {tuning_results['best_separation']:.6f}")

# Visualização: γ vs separação
results_df = pd.DataFrame(tuning_results["results"])
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(results_df["gamma"], results_df["separation"], marker="o", color="steelblue")
ax.axvline(tuning_results["best_gamma"], color="red", linestyle="--",
           label=f"Melhor γ = {tuning_results['best_gamma']:.4f}")
ax.set_xlabel("γ (parâmetro do kernel RBF)")
ax.set_ylabel("Separação (MMD observada − média nula)")
ax.set_title("Busca de γ – Maximização da Separação (Documento 04)")
ax.set_xscale("log")
ax.legend()
plt.tight_layout()
plt.show()

## 9. Salvando o Detector

Treinamos o `DriftDetector` com os dados de referência e persistimos o modelo
e as métricas calculadas para uso no **Notebook 03** (comparação em produção
e integração com Alibi Detect), conforme fluxo descrito no **Documento 04**.

In [ ]:
# Treina o detector completo
detector = train_model(X_ref, gamma=tuning_results["best_gamma"])

# Salva modelo e métricas
model_path = save_model(detector)
metrics_path = save_metrics(metrics)

print(f"Modelo salvo em:   {model_path}")
print(f"Métricas salvas em: {metrics_path}")
print("\n→ Artefatos prontos para o Notebook 03.")

## Resumo

Neste notebook (correspondente aos **Vídeos 2 e 3 do Documento 04**), implementamos
e comparamos métricas avançadas de detecção de drift:

| Métrica | Tipo | Detecta Drift Multivariado? |
|---------|------|-----------------------------|
| PSI | Univariada | ❌ Não (marginais inalteradas) |
| MMD (kernel RBF) | Multivariada | ✅ Sim |
| Wasserstein | Univariada (por feature) | ⚠️ Parcialmente |
| Energy Distance | Multivariada | ✅ Sim |

**Principais conclusões** (Documento 04):
- O **PSI falha** para drift multivariado (inversão de correlação renda–idade).
- **MMD** e **Energy Distance** detectam mudanças na distribuição conjunta.
- A **mediana heuristic** fornece um γ robusto para o kernel RBF.
- A **validação cruzada** permite calibrar limiares sem dados rotulados.

### Próximo Passo

No **Notebook 03**, faremos a comparação final em cenário de produção e integraremos
com a biblioteca **Alibi Detect** para validação cruzada das implementações.